In [1]:
import os

import numpy as np
import pandas as pd
import xarray as xr

from utils import PROCESSED_DIR

# Directory containing Evangelou2026 data
DATA_DIR = "data/original"

# Overwrite outputs
OVERWRITE = True


def prepare_evangelou2026(
    data_dir: str, out_name: str, out_dir: str = PROCESSED_DIR, overwrite: bool = False
) -> pd.DataFrame:
    """Load Evangelou2026 dataset and fix typos / data errors"""

    # Skip if processed unless overwrite
    out_path = os.path.join(out_dir, out_name)
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return pd.read_csv(out_path, parse_dates=["Start date", "End date"])

    # Load raw data
    data = pd.read_excel(os.path.join(data_dir, "evangelou2026.xlsx"))
    data.columns = data.columns.str.strip()

    # Clean author
    data["Author"] = data["Author"].str.strip()

    # Fill and homogenize DOI
    data["DOI"] = (
        data["DOI"]
        .str.strip()
        .str.lower()
        .str.replace("http://", "https://")
        .str.replace("dx.doi.org", "doi.org")
        .str.replace("www.science.org/doi", "doi.org")
        .ffill()
    )

    # Homogenize length
    data["Length (um)"] = (
        data["Length (um)"]
        .str.replace(" -", "-")
        .str.replace(" to ", "-")
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"(\d),(>|\d)", r"\1, \2", regex=True)
        .str.replace(r"^,", "", regex=True)
    )

    # Clean shape
    data["Shape"] = data["Shape"].str.strip()

    # Fix typos / data errors
    data = revise_evangelou2026(data)

    # Ensure dates are datetimes
    data["Start date"] = pd.to_datetime(data["Start date"])
    data["End date"] = pd.to_datetime(data["End date"])

    # Save to disk
    data.to_csv(out_path, index=False)
    print(f"-> {out_path}")

    return data


def revise_evangelou2026(data: pd.DataFrame) -> pd.DataFrame:
    """Fix typos / data errors"""

    # Temporarily rename columns for convenience
    data[["edits", "comments"]] = None
    orig_cols = data.columns
    data = data.rename(columns={"lat (north)": "lat", "lon (east)": "lon"})

    def get_mask(doi: str) -> pd.Series:
        return data["DOI"].eq(doi)

    def update(mask: pd.Series | pd.Index, updates: dict) -> None:
        for col in ["edits", "comments"]:
            if col in updates:
                old = data.loc[mask, col]
                present = old.notna()
                new = pd.Series(updates[col], index=old.index)
                new.loc[present] = old.loc[present] + "; " + new.loc[present]
                updates[col] = new.to_list()
        for k, v in updates.items():
            data.loc[mask, k] = v

    # Cai2017
    # Exclude non-plastic fibers
    cai2017 = get_mask("https://doi.org/10.1007/s11356-017-0116-x")
    parts = [67, 56, 52, 44, 42, 41, 40, 39, 45]
    mg = (
        parts
        / data.loc[cai2017, "parts/m2/day (bulk)"]
        * data.loc[cai2017, "mg/m2/day (bulk)"]
    )
    update(
        mask=cai2017,
        updates={
            "parts/m2/day (bulk)": parts,
            "mg/m2/day (bulk)": mg,
            "edits": "adjusted parts and mg to exclude non-plastic fibers",
        },
    )

    # Dris2015
    # Remove as duplicate of Dris2017 but with shorter sampling period
    data = data.loc[~get_mask("https://doi.org/10.1071/en14167"), :]

    # Allen2019
    # Fix day-first dates
    starts = 3 * ["2017-11-16", "2017-11-29", "2017-12-19", "2018-01-23", "2018-03-06"]
    ends = 3 * ["2017-11-28", "2017-12-18", "2018-01-22", "2018-03-05", "2018-04-09"]
    update(
        mask=get_mask("https://doi.org/10.1038/s41561-019-0335-5"),
        updates={
            "Start date": starts,
            "End date": ends,
            "edits": "fixed misformatted start and end dates",
        },
    )
    data["Start date"] = pd.to_datetime(data["Start date"])
    data["End date"] = pd.to_datetime(data["End date"])

    # Brahney2020
    # Fix locations and some dates
    brahney2020 = get_mask("https://doi.org/10.1126/science.aaz5819")
    update(
        mask=brahney2020 & data["lat"].eq(40.34) & data["lon"].eq(-105.67),
        updates={
            "lat": 40.2878,
            "lon": -105.6628,
            "edits": "adjusted location to match NADP station CO98",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(36.11) & data["lon"].eq(-112.1),
        updates={
            "lat": 36.0586,
            "lon": -112.184,
            "edits": "adjusted location to match NADP station AZ03",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(43.18) & data["lon"].eq(-109.65),
        updates={
            "lat": 42.929,
            "lon": -109.7875,
            "edits": "adjusted location to match NADP station WY06",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(37.6) & data["lon"].eq(-112.18),
        updates={
            "lat": 37.6186,
            "lon": -112.1728,
            "edits": "adjusted location to match NADP station UT99",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(38.8) & data["lon"].eq(-106.91),
        updates={
            "lat": 38.9561,
            "lon": -106.986,
            "edits": "adjusted location to match NADP station CO10",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(43.46) & data["lon"].eq(-113.53),
        updates={
            "lat": 43.4605,
            "lon": -113.5551,
            "edits": "adjusted location to match NADP station ID03",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(38.93) & data["lon"].eq(-114.25),
        updates={
            "lat": 39.0054,
            "lon": -114.217,
            "edits": "adjusted location to match NADP station NV05",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(33.86) & data["lon"].eq(-115.89),
        updates={
            "lat": 34.0695,
            "lon": -116.3889,
            "edits": "adjusted location to match NADP station CA67",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(38.21) & data["lon"].eq(-109.89),
        updates={
            "lat": 38.4584,
            "lon": -109.821,
            "edits": "adjusted location to match NADP station UT09",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(40.079) & data["lon"].eq(-105.57),
        updates={
            "lat": 40.0547,
            "lon": -105.5891,
            "edits": "adjusted location to match NADP station CO02",
        },
    )
    update(
        mask=brahney2020 & data["lat"].eq(40.735) & data["lon"].eq(-110.5),
        updates={
            "lat": 40.7543,
            "lon": -109.4671,
            "edits": "adjusted location to match NADP station UT95",
        },
    )
    update(
        mask=(
            brahney2020
            & data["lat"].eq(38.9561)
            & data["lon"].eq(-106.986)
            & data["Shape"].eq("sphere")
            & data["Start date"].eq("2018-12-25")
            & data["End date"].eq("2018-12-25")
        ),
        updates={
            "Start date": "2018-12-18",
            "edits": "adjusted start date based on supplementary data S4",
        },
    )

    # Gaston2020
    # Use Nile red counts rather than visual counts
    gaston2020 = get_mask("https://doi.org/10.1177/0003702820920652")
    parts = [0.6, 5.6]
    mg = (
        parts
        / data.loc[gaston2020, "parts/m3 (air)"]
        * data.loc[gaston2020, "mg/m3 (air)"]
    )
    update(
        mask=gaston2020,
        updates={
            "parts/m3 (air)": parts,
            "mg/m3 (air)": mg,
            "edits": "adjusted parts and mg to use counts based on Nile red staining",
        },
    )

    # Jenner2022
    # Mislabeled as Abbasi2019
    update(
        mask=data.loc[845:870].index,
        updates={
            "Author": "Jenner",
            "Year of publication": 2022,
            "DOI": "https://doi.org/10.3390/atmos13020265",
            "edits": "fixed author, year, and DOI",
        },
    )

    # Liu2022
    # Add dry deposition data
    liu2022 = get_mask("https://doi.org/10.1016/j.ecoenv.2022.113353")
    parts = [95.3088, 39.6435, 42.8355, 89.0886, 37.0562, 40.0399]
    mg = (
        parts
        / data.loc[liu2022, "parts/m2/day (wet)"]
        * data.loc[liu2022, "mg/m2/day (wet)"]
    )
    dry_dep = (
        data.loc[liu2022, :]
        .copy()
        .assign(
            **{
                "Dry deposition": "o",
                "Wet deposition": None,
                "parts/m2/day (dry)": parts,
                "parts/m2/day (wet)": None,
                "mg/m2/day (dry)": mg,
                "mg/m2/day (wet)": None,
                "edits": "added dry deposition data",
            }
        )
        .dropna(axis="columns")
    )
    data = pd.concat([data, dry_dep], ignore_index=True)

    # Abbasi2021
    # Convert monthly data to daily
    abbasi2021 = get_mask("https://doi.org/10.1016/j.scitotenv.2021.147358")
    n_days = (
        pd.to_datetime(data.loc[abbasi2021, "End date"])
        - pd.to_datetime(data.loc[abbasi2021, "Start date"])
    ).dt.days + 1
    update(
        mask=abbasi2021,
        updates={
            "parts/m2/day (dry)": data.loc[abbasi2021, "parts/m2/day (dry)"] / n_days,
            "parts/m2/day (wet)": data.loc[abbasi2021, "parts/m2/day (wet)"] / n_days,
            "mg/m2/day (dry)": data.loc[abbasi2021, "mg/m2/day (dry)"] / n_days,
            "mg/m2/day (wet)": data.loc[abbasi2021, "mg/m2/day (wet)"] / n_days,
            "edits": "adjusted parts and mg from per month to per day",
        },
    )

    # Liao2021
    # Mislabaled as Gonzalez2021; remove duplicate rows correctly labelled as Liao2021
    data = data.loc[~get_mask("https://doi.org/10.1016/j.jhazmat.2021.126007"), :]
    update(
        mask=data.loc[1360:1365].index,
        updates={
            "Author": "Liao",
            "Year of publication": 2021,
            "DOI": "https://doi.org/10.1016/j.jhazmat.2021.126007",
            "edits": "fixed author, year, and DOI",
        },
    )

    # Allen2022
    # Remove as duplicate of Allen2019. The data here do not match b/c Allen2022 estimated
    # the mass corresponding to the Allen2019 MP counts and Evangelou2026 back-calculated
    # MP counts from the reported masses.
    data = data.loc[~get_mask("https://doi.org/10.1016/j.hazadv.2022.100104"), :]

    # Yuan2023
    # Fix sampling height (sampled on Canton tower; height is above *ground level*)
    yuan2023 = get_mask("https://doi.org/10.1016/j.scitotenv.2023.165190")
    update(
        mask=yuan2023 & data["height (m asl)"].eq(118),
        updates={"height (m asl)": 0, "edits": "adjusted height"},
    )
    update(mask=yuan2023, updates={"comments": "height is above ground level"})

    # Reset index and restore column names
    data.columns = orig_cols
    data = data.rename_axis("idx_orig", axis="index").reset_index(drop=False)

    return data


Introduction
------------

This notebook processes the dataset of atmospheric microplastic observations collected by @Evangelou2026. We use these alternate data to evaluate the sensitivity of our main results to the observational dataset used to constrain simulated atmospheric microplastic emissions.

## Revise observations

We correct a few typos and errors in the original dataset and remove two duplicate studies (Dris2015 is duplicate of Dris2016 but with shorter sampling period; Allen2022 is duplicate of Allen2019).

In [2]:
evangelou2026 = prepare_evangelou2026(
    data_dir=DATA_DIR, out_name="evangelou2026-revised.csv", overwrite=OVERWRITE
)

-> data/processed/evangelou2026-revised.csv


## Aggregate by location

We then compute the mean microplastic concentration or bulk deposition for each sampling location and reported microplastic shape (fragments, fibers, films, spheres).

In [3]:
# Make single column with all measures
evangelou2026["measure"] = "dry_deposition"
evangelou2026["number"] = evangelou2026["parts/m2/day (dry)"]
evangelou2026["number_units"] = "particle/m2/d"

wet = evangelou2026["Wet deposition"].notna()
evangelou2026.loc[wet, "measure"] = "wet_deposition"
evangelou2026.loc[wet, "number"] = evangelou2026.loc[wet, "parts/m2/day (wet)"]

bulk = evangelou2026["Bulk deposition"].notna()
evangelou2026.loc[bulk, "measure"] = "deposition"
evangelou2026.loc[bulk, "number"] = evangelou2026.loc[bulk, "parts/m2/day (bulk)"]

conc = evangelou2026["Air concentration"].notna()
evangelou2026.loc[conc, "measure"] = "concentration"
evangelou2026.loc[conc, "number"] = evangelou2026.loc[conc, "parts/m3 (air)"]
evangelou2026.loc[conc, "number_units"] = "particle/m3"

# Estimate sampling duration of each observation
# Fix duration for:
# - Abbasi2021 (10.1016/j.hazadv.2021.100035) -> sampled for 30 min during monsoon rain
# - Li2020 (10.1016/j.scitotenv.2019.135967) -> sampled for 4/8 hours
evangelou2026["duration"] = evangelou2026["End date"] - evangelou2026["Start date"]
abbasi2021 = evangelou2026["DOI"].eq("https://doi.org/10.1016/j.hazadv.2021.100035")
evangelou2026.loc[abbasi2021, "duration"] = pd.to_timedelta([10, 10, 10], "minutes")
li2020 = evangelou2026["DOI"].eq("https://doi.org/10.1016/j.scitotenv.2019.135967")
evangelou2026.loc[li2020, "duration"] = pd.to_timedelta([4, 4, 8, 8], "hours")


# Compute sampling duration-weighted mean per location and MP shape
# Ok to combine:
# - Length average for Abbasi2021 SciTotEnv, Amato2022, Kaushik2024, Morioka2025,
#     Parashar2023
# - Length (um) for Gossman2023
# fmt: off
group_cols = [
    "Author", "Year of publication", "DOI", "Shape", "measure", "lat (north)",
    "lon (east)", "height (m asl)"
]
# fmt: on
obs = (
    evangelou2026.groupby(group_cols, as_index=False, dropna=False)[
        ["Start date", "End date", "number", "number_units", "duration"]
    ]
    .apply(
        lambda x: pd.Series(
            {
                "Start date": x["Start date"].min(),
                "End date": x["End date"].max(),
                "duration": x["duration"].sum(),
                "number": np.average(
                    x["number"], weights=x["duration"].dt.total_seconds()
                ),
                "number_units": x["number_units"].iloc[0],
                "n_obs": len(x),
            }
        )
    )
    .rename(
        columns={
            "Year of publication": "year",
            "lat (north)": "lat",
            "lon (east)": "lon",
            "height (m asl)": "height",
        }
    )
    .rename(columns=lambda x: x.lower().replace(" ", "_"))
)

# Summarize
obs.groupby(["measure", "shape"]).agg(
    studies=("doi", "nunique"),
    locations=(
        "lat",
        lambda x: obs.loc[x.index, ["lat", "lon"]].drop_duplicates().shape[0],
    ),
    observations=("number", "count"),
    number=("number", "mean"),
    number_units=("number_units", "first"),
).round(1)

studies  locations  observations  number  \
measure        shape                                                
concentration  fiber          30        159           159    40.3   
               film            3         24            24     0.0   
               fragment       22        120           120    14.5   
               sphere          8         28            30    36.1   
deposition     fiber          27         69            69   122.1   
               film            7         12            12   220.5   
               fragment       21         52            53    72.7   
               sphere          6         10            10    84.1   
dry_deposition fiber           5         16            16   186.3   
               film            2          2             2   250.8   
               fragment        3          3             3    24.9   
               sphere          2         12            12    27.9   
wet_deposition fiber           9         23            23   694.9   
               film            3          4             4   215.1   
               fragment        5          6             6    43.9   
               sphere          2         12            12    34.2   

                          number_units  
measure        shape                    
concentration  fiber       particle/m3  
               film        particle/m3  
               fragment    particle/m3  
               sphere      particle/m3  
deposition     fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d  
dry_deposition fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d  
wet_deposition fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d

## Exclude obs high above ground level

We exclude observations sampled high above ground level (Gonzalez-Pleiter2021, Yuan2023 observations above ground level).

In [4]:
obs = obs.loc[
    ~obs["doi"].eq("https://doi.org/10.1016/j.scitotenv.2020.143213")
    & ~(
        obs["doi"].eq("https://doi.org/10.1016/j.scitotenv.2023.165190")
        & obs["height"].gt(0)
    )
].reset_index(drop=True)
obs.loc[obs["doi"].eq("https://doi.org/10.1016/j.scitotenv.2023.165190"), "height"] = None

# Summarize
obs.groupby(["measure", "shape"]).agg(
    studies=("doi", "nunique"),
    locations=(
        "lat",
        lambda x: obs.loc[x.index, ["lat", "lon"]].drop_duplicates().shape[0],
    ),
    observations=("number", "count"),
    number=("number", "mean"),
    number_units=("number_units", "first"),
).round(1)

studies  locations  observations  number  \
measure        shape                                                
concentration  fiber          29        152           152    42.0   
               film            3         24            24     0.0   
               fragment       21        113           113    12.8   
               sphere          8         28            28    38.7   
deposition     fiber          27         69            69   122.1   
               film            7         12            12   220.5   
               fragment       21         52            53    72.7   
               sphere          6         10            10    84.1   
dry_deposition fiber           5         16            16   186.3   
               film            2          2             2   250.8   
               fragment        3          3             3    24.9   
               sphere          2         12            12    27.9   
wet_deposition fiber           9         23            23   694.9   
               film            3          4             4   215.1   
               fragment        5          6             6    43.9   
               sphere          2         12            12    34.2   

                          number_units  
measure        shape                    
concentration  fiber       particle/m3  
               film        particle/m3  
               fragment    particle/m3  
               sphere      particle/m3  
deposition     fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d  
dry_deposition fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d  
wet_deposition fiber     particle/m2/d  
               film      particle/m2/d  
               fragment  particle/m2/d  
               sphere    particle/m2/d

## Merge dry and wet deposition

For studies that sampled dry or wet deposition in addition to bulk deposition (Roblin2020, Szewc2021, Welsh2022), we use only the bulk deposition samples. For studies that sampled dry and wet deposition separately (Abbasi2021 SciTotEnv, Brahney2021, Jia2022, Kernchen2022, Liu2022), we combine these to get total deposition. We exclude studies that sampled only dry or wet deposition (Abbasi2021 JHazAdv, Parashar2023).

In [5]:
# Combine dry + wet deposition -> total deposition
# Exclude Szewc wet + dry b/c also sampled total deposition
dry_wet = obs["measure"].isin(["dry_deposition", "wet_deposition"])
total_dois = (
    obs.loc[dry_wet]
    .groupby(["author", "doi", "shape", "lat", "lon"], as_index=False)[["measure"]]
    .nunique()
    .query("measure == 2 & author != 'Szewc'")["doi"]
    .unique()
)

group_cols = ["author", "year", "doi", "shape", "lat", "lon", "height"]
tot_dep = (
    obs.loc[dry_wet & obs["doi"].isin(total_dois)]
    .groupby(group_cols, as_index=False, dropna=False)
    .agg(
        {
            "measure": lambda x: "deposition",
            "number_units": "first",
            "start_date": "min",
            "end_date": "max",
            "duration": "max",
            "number": "sum",
            "n_obs": "max",
        }
    )
)

sort_cols = ["author", "year", "doi", "shape", "measure", "lat", "lon"]
obs = (
    pd.concat([obs.loc[~dry_wet], tot_dep]).sort_values(sort_cols).reset_index(drop=True)
)

# fmt: off
col_order = [
    "author", "year", "doi", "lat", "lon", "height", "start_date", "end_date", "duration",
    "shape", "measure", "number", "number_units", "n_obs"
]
# fmt: on
obs = obs[col_order]

# Summarize
obs.groupby(["measure", "shape"]).agg(
    studies=("doi", "nunique"),
    locations=(
        "lat",
        lambda x: obs.loc[x.index, ["lat", "lon"]].drop_duplicates().shape[0],
    ),
    observations=("number", "count"),
    number=("number", "mean"),
    number_units=("number_units", "first"),
).round(1)

studies  locations  observations  number  \
measure       shape                                                
concentration fiber          29        152           152    42.0   
              film            3         24            24     0.0   
              fragment       21        113           113    12.8   
              sphere          8         28            28    38.7   
deposition    fiber          31         84            84   174.4   
              film            8         13            13   298.5   
              fragment       23         53            55    73.2   
              sphere          8         22            22    72.1   

                         number_units  
measure       shape                    
concentration fiber       particle/m3  
              film        particle/m3  
              fragment    particle/m3  
              sphere      particle/m3  
deposition    fiber     particle/m2/d  
              film      particle/m2/d  
              fragment  particle/m2/d  
              sphere    particle/m2/d

## Aggregate by model grid cell

For studies that reported data sampled at multiple locations, we aggregate all observations that fall within the same model grid cell.

In [6]:
grid = xr.open_dataset("results/sim_main.nc")[["lat", "lon"]]
obs["grid_lat"] = grid["lat"].sel(lat=obs["lat"].to_numpy(), method="nearest")
obs["grid_lon"] = grid["lon"].sel(lon=obs["lon"].to_numpy(), method="nearest")

# fmt: off
group_cols = [
    "author", "year", "doi", "shape", "measure", "height", "grid_lat", "grid_lon"
]
agg_cols = [
    "lat", "lon", "start_date", "end_date", "duration", "number", "number_units", "n_obs"
]
# fmt: on
obs = obs.groupby(group_cols, as_index=False, dropna=False)[agg_cols].apply(
    lambda x: pd.Series(
        {
            "lat": np.average(x["lat"], weights=x["duration"].dt.total_seconds()),
            "lon": np.average(x["lon"], weights=x["duration"].dt.total_seconds()),
            "start_date": x["start_date"].min(),
            "end_date": x["end_date"].max(),
            "duration": x["duration"].sum(),
            "number": np.average(x["number"], weights=x["duration"].dt.total_seconds()),
            "number_units": x["number_units"].iloc[0],
            "n_obs": x["n_obs"].sum(),
            "n_loc": len(x),
        }
    )
)

# Summarize
obs.groupby(["measure", "shape"]).agg(
    studies=("doi", "nunique"),
    locations=(
        "lat",
        lambda x: obs.loc[x.index, ["lat", "lon"]].drop_duplicates().shape[0],
    ),
    observations=("number", "count"),
    number=("number", "mean"),
    number_units=("number_units", "first"),
).round(1)

studies  locations  observations  number  \
measure       shape                                                
concentration fiber          29         96            96    63.1   
              film            3          9             9     0.1   
              fragment       21         86            86    12.6   
              sphere          8         18            18    42.2   
deposition    fiber          31         55            55   215.7   
              film            8          9             9   398.8   
              fragment       23         36            37    92.4   
              sphere          8         17            17    62.5   

                         number_units  
measure       shape                    
concentration fiber       particle/m3  
              film        particle/m3  
              fragment    particle/m3  
              sphere      particle/m3  
deposition    fiber     particle/m2/d  
              film      particle/m2/d  
              fragment  particle/m2/d  
              sphere    particle/m2/d

## Drop null observations

We exclude any observations that found no microplastics (concentration or deposition = 0).

In [7]:
obs = (
    obs.loc[obs["number"].gt(0)]
    .reset_index(drop=True)
    .sort_values(["measure", "author", "year"])
)

# Summarize
obs.groupby(["measure", "shape"]).agg(
    studies=("doi", "nunique"),
    locations=(
        "lat",
        lambda x: obs.loc[x.index, ["lat", "lon"]].drop_duplicates().shape[0],
    ),
    observations=("number", "count"),
    number=("number", "mean"),
    number_units=("number_units", "first"),
).round(1)

studies  locations  observations  number  \
measure       shape                                                
concentration fiber          29         81            81    74.8   
              film            3          7             7     0.1   
              fragment       21         71            71    15.3   
              sphere          8         18            18    42.2   
deposition    fiber          31         55            55   215.7   
              film            8          9             9   398.8   
              fragment       23         36            37    92.4   
              sphere          8         17            17    62.5   

                         number_units  
measure       shape                    
concentration fiber       particle/m3  
              film        particle/m3  
              fragment    particle/m3  
              sphere      particle/m3  
deposition    fiber     particle/m2/d  
              film      particle/m2/d  
              fragment  particle/m2/d  
              sphere    particle/m2/d

## Save

We save the aggregated observations to disk.

In [8]:
# Skip if processed unless overwrite
out_path = os.path.join(PROCESSED_DIR, "obs_evangelou2026-revised.csv")
if not os.path.exists(out_path) or OVERWRITE:
    obs.to_csv(out_path, index=False)
    print(f"-> {out_path}")
else:
    print(f"File exists: {out_path}")

# Print for reference
obs

-> data/processed/obs_evangelou2026-revised.csv


,author,year,doi,shape,measure,height,grid_lat,grid_lon,lat,lon,start_date,end_date,duration,number,number_units,n_obs,n_loc
0,Abbasi,2019,https://doi.org/10.1016/j.envpol.2018.10.039,fiber,concentration,NaN,28.0,52.5,27.310000,52.330000,2017-07-31,2017-08-28,8 days,1.155695,particle/m3,8,1
2,Abbasi,2023,https://doi.org/10.1016/j.jes.2022.02.044,fiber,concentration,NaN,32.0,47.5,31.417333,48.672000,2019-08-28,2019-12-23,25 days,0.006872,particle/m3,25,2
3,Akhbarizadeh,2021,https://doi.org/10.1016/j.envres.2020.110339,fiber,concentration,NaN,28.0,50.0,28.979600,50.834300,2017-01-04,2017-09-20,12 days,1.325833,particle/m3,12,1
7,Allen,2020,https://doi.org/10.1371/journal.pone.0232746,sphere,concentration,NaN,44.0,-2.5,44.216667,-1.283333,2018-10-05,2018-10-13,8 days,6.265435,particle/m3,8,1
8,Allen,2021,https://doi.org/10.1038/s41467-021-27454-7,fiber,concentration,NaN,42.0,0.0,42.936802,0.141170,2017-06-23,2017-10-23,97 days,0.070233,particle/m3,15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,Zhang,2023,https://doi.org/10.1016/j.scitotenv.2023.163567,fiber,deposition,NaN,40.0,117.5,40.100000,116.500000,2021-09-01,2022-02-28,360 days,248.108206,particle/m2/d,2,2
287,Zhang,2023,https://doi.org/10.1016/j.scitotenv.2023.163567,film,deposition,NaN,40.0,115.0,39.966667,115.433333,2021-09-01,2022-02-28,180 days,6.818816,particle/m2/d,1,1
288,Zhang,2023,https://doi.org/10.1016/j.scitotenv.2023.163567,film,deposition,NaN,40.0,117.5,40.100000,116.500000,2021-09-01,2022-02-28,360 days,14.724864,particle/m2/d,2,2
289,Zhang,2023,https://doi.org/10.1016/j.scitotenv.2023.163567,fragment,deposition,NaN,40.0,115.0,39.966667,115.433333,2021-09-01,2022-02-28,180 days,9.828684,particle/m2/d,1,1
